In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install -q transformers accelerate sentencepiece


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

# ---------------------------
# Configuration
# ---------------------------
MODEL_NAME = "microsoft/phi-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

# ---------------------------
# Load knowledge base
# ---------------------------
kb_path = Path("/content/maintenance_guidelines.txt")
knowledge_text = kb_path.read_text()

# ---------------------------
# Load model & tokenizer
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

print("Model loaded successfully")


Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully


In [21]:
def ai_engineer_assistant(features: dict, prediction: dict):
    prompt = f"""
You are an AI Maintenance Engineer.

Machine sensor data:
{features}

ML model output:
Failure probability: {prediction['failure_probability']}

Maintenance knowledge:
{knowledge_text}

Explain clearly:
1. Current machine health
2. Possible failure mode (if any)
3. Recommended maintenance action
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    output = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,
        do_sample=True
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [22]:
sample_features = {
    "Air temperature": 300,
    "Process temperature": 311,
    "Rotational speed": 1400,
    "Torque": 45,
    "Tool wear": 180,
    "Type": "M"
}

sample_prediction = {
    "failure_probability": 0.73,
    "prediction": 1
}

response = ai_engineer_assistant(sample_features, sample_prediction)
print(response)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



You are an AI Maintenance Engineer.

Machine sensor data:
{'Air temperature': 300, 'Process temperature': 311, 'Rotational speed': 1400, 'Torque': 45, 'Tool wear': 180, 'Type': 'M'}

ML model output:
Failure probability: 0.73

Maintenance knowledge:
Predictive Maintenance Guidelines

1. High temperature difference combined with high tool wear
   indicates potential heat dissipation failure.

2. High power consumption outside normal operating range
   may indicate power failure or mechanical resistance.

3. High torque combined with excessive tool wear
   suggests overstrain failure.

4. Low failure probability (<0.2) typically indicates
   healthy machine operation.

5. Preventive maintenance is recommended when failure
   probability exceeds 0.6.


Explain clearly:
1. Current machine health
2. Possible failure mode (if any)
3. Recommended maintenance action

Question:
Considering the provided data and the ML model output, what is the current health of the machine? If any, what is the

In [7]:
%pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.6 MB/s eta 0:00:00


In [23]:
from pathlib import Path

kb_path = Path("/content/maintenance_guidelines.txt")
text = kb_path.read_text()

# Simple chunking
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

chunks = chunk_text(text)

print(f"Total chunks: {len(chunks)}")


Total chunks: 1


In [24]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)
embeddings = np.array(embeddings)

print("Embeddings shape:", embeddings.shape)


Embeddings shape: (1, 384)


In [25]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 1


In [26]:
def retrieve_relevant_context(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    unique_chunks = []
    for i in indices[0]:
        if chunks[i] not in unique_chunks:
            unique_chunks.append(chunks[i])

    return "\n".join(unique_chunks)


In [49]:
def ai_engineer_assistant_rag(features: dict, prediction: dict):
    """
    RAG-powered AI Maintenance Engineer Assistant
    Inputs:
        features   : dict of machine sensor values
        prediction : dict with failure_probability
    Output:
        Plain English maintenance assessment
    """

    # Step 1: Build retrieval query
    query = f"""
    Machine failure probability is {prediction['failure_probability']}.
    Tool wear is {features['Tool wear']}.
    Temperature difference is {features['Process temperature'] - features['Air temperature']}.
    """

    # Step 2: Retrieve relevant maintenance knowledge
    retrieved_context = retrieve_relevant_context(query)

    # Step 3: Construct controlled prompt
    prompt = f"""
You are a senior industrial maintenance engineer.

IMPORTANT:
Your response MUST start with a sentence describing machine health.
If you write code, the answer is invalid.

INSTRUCTIONS (MANDATORY):
- Write ONE professional paragraph
- Do NOT explain your reasoning
- Do NOT analyze parameters individually
- Do NOT restate sensor values or units
- Do NOT add headings or labels
- Do NOT reference guidelines by number
- Do NOT write code
- Do NOT mention models, algorithms, or libraries
- Use plain English only
- Maximum 100 words


Machine condition summary:
{features}

ML model output:
Failure probability = {prediction['failure_probability']}

Maintenance reference:
{retrieved_context}

FINAL ANSWER (TEXT ONLY):
"""

    # Step 4: Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Step 5: Generate response
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.4,
        do_sample=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

    # Step 6: Decode
    decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # Step 7: Extract final answer only
    final_answer = decoded_output.split(
        "FINAL ANSWER (TEXT ONLY):"
    )[-1].strip()

    # Step 8: Safety filter (block code generation)
    forbidden_tokens = ["import ", "def ", "class ", "pd.", "sklearn", "python"]

    for token in forbidden_tokens:
        if token in final_answer.lower():
            return (
                "The machine shows an elevated risk of failure and is not operating "
                "under optimal conditions. Preventive maintenance is strongly "
                "recommended to reduce the likelihood of unplanned downtime."
            )

    return final_answer


In [50]:
sample_features = {
    "Air temperature": 300,
    "Process temperature": 311,
    "Rotational speed": 1400,
    "Torque": 45,
    "Tool wear": 180,
    "Type": "M"
}

sample_prediction = {
    "failure_probability": 0.73
}

print(ai_engineer_assistant_rag(sample_features, sample_prediction))


Based on the provided data and ML model's prediction of a 73% chance for failure within 24 hours, it is advisable to implement preventive maintenance measures as soon as possible. The combination of an air temperature above room temperature, process and rotational speeds at their maximum limits, along with excessively worn tools and a type M machine, all point towards potential failures related to overheating, inadequate cooling systems, or mechanical strain. While there is still some uncertainty due to the relatively low failure probability, it would be prudent not to take any chances in order to avoid unexpected breakdowns that could lead to costly downtime and
